<a href="https://colab.research.google.com/github/Okahak71/learning_ai/blob/main/Transformer/For_transformers_day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn.functional as F
import torch.nn as nn

In [22]:
#Hyperparameters
B = 128
n = 16
d = 256

In [ ]:
def single_attention(x, wq, wk, wv):
    b, t, d = x.shape
    print(f"shape x : {x.shape}, \nshape wq : {wq.shape}")
    q, k, v = x @ wq, x @ wk, x @ wv
    sz = q.shape[-1]

    #computation
    s = q @ k.transpose(-2, -1) / sz**0.5 #(B, T, T)
    mask = torch.triu(torch.ones(t, t, device=x.device), diagonal=1).bool()
    s = s.masked_fill(mask, float('-inf'))

    return (torch.softmax(s, dim = -1)) @ v #(B, T, T) @ (B, T, d_k) -> (B, T, d_k)

In [ ]:
def multi_head(x, W, n):
    B, T, d = x.shape; d_k = d // n
    Wq, Wk, Wv, Wo = W
    print(f"shape wq : {Wq.shape}")
    #single attention -> (B, T, d_k)
    Q = (x @ Wq).view(B, T, n, d_k).transpose(1, 2)
    K = (x @ Wk).view(B, T, n, d_k).transpose(1, 2)
    V = (x @ Wv).view(B, T, n, d_k).transpose(1, 2)
    s = Q @ K.transpose(-2, -1) / d_k**0.5
    mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
    s = s.masked_fill(mask, float('-inf'))
    res = torch.softmax(s, dim = -1) @ V
    res = res.transpose(1, 2)
    print(f"shape -> {res.shape}")
    return res.reshape(B, T, -1) @ Wo

In [23]:
class MHA(nn.Module):
    def __init__(self, d, n):
        super().__init__()
        self.d = d; self.n = n
        self.wq = nn.Linear(d, d, bias = False)
        self.wk = nn.Linear(d, d, bias = False)
        self.wv = nn.Linear(d, d, bias = False)
        self.wo = nn.Linear(d, d, bias = False)
    def forward(self, x):
        B, T = x.shape[:2] #(3, 5)
        d_k = self.d // self.n # 8
        Q = self.wq(x).view(B, T, self.n, d_k).transpose(1, 2)
        K = self.wk(x).view(B, T, self.n, d_k).transpose(1, 2)
        V = self.wv(x).view(B, T, self.n, d_k).transpose(1, 2)
        s = Q @ K.transpose(-2, -1) / d_k**0.5
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        s = s.masked_fill(mask, float('-inf'))
        return self.wo((torch.softmax(s, dim = -1) @ V).transpose(1, 2).reshape(B, T, -1))

In [ ]:
def multi_head_loop(x, Wq, Wk, Wv, Wo, n):
    d_k = Wq.shape[1] // n
    sl = lambda W, i: W[:, i*d_k:(i+1)*d_k]     # head i = columns i*d_k ... (i+1)*d_k
    heads = [single_attention(x, sl(Wq,i), sl(Wk,i), sl(Wv,i)) for i in range(n)]
    return torch.cat(heads, dim=-1) @ Wo

In [24]:
class MyLayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d))    # shape (d,)
        self.beta  = nn.Parameter(torch.zeros(d))   # shape (d,)
    def forward(self, x):
        mean = x.mean(dim = -1, keepdim = True)
        var = x.var(dim = -1, correction=0, keepdim = True)
        xx = (x - mean) / (var + self.eps)**0.5
        return self.gamma * xx + self.beta

In [25]:
class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(d, 4 * d),
            nn.GELU(),
            nn.Linear(4 * d, d)
        )
    def forward(self, x):
        return self.layers(x)


def mlp_manual(x, w1, b1, w2, b2):
    xpact = F.gelu(x @ w1 + b1)
    return xpact @ w2 + b2

In [28]:
x = torch.randn(B, 200, d);
# # x = x.view(B, T, n, -1).transpose(1, 2)
# mha = MHA(d, n)
# W = [mha.layers[i].weight.T for i in range(4)]
# out_vec = multi_head(x, W, n)
# out_class = mha(x)
# print(torch.allclose(out_vec, out_class, atol=1e-4))   # float rounding, so use a tolerance

In [31]:
class Block(nn.Module):
    def __init__(self, d, n):
        super().__init__()
        self.ln1, self.ln2 = MyLayerNorm(d), MyLayerNorm(d)
        self.attn, self.mlp = MHA(d, n), MLP(d)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

blk = Block(d, n)
y = blk(x)
assert torch.equal(blk(x), blk(x))